In [9]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fineas.io.paths import (
    norm_ohlcv_root,
    norm_ohlcv_part_dir,
    norm_ohlcv_part_path,
    raw_yf_ohlcv_part_path,
)

In [ ]:
# Phase 2A: Normalized OHLCV

In [2]:
symbol = "AAPL"
interval = "1d"
year = 2022
month = 1

raw_p = raw_yf_ohlcv_part_path(symbol=symbol, interval=interval, year=year, month=month, repo_root=ROOT)
norm_p = norm_ohlcv_part_path(symbol=symbol, interval=interval, year=year, month=month, repo_root=ROOT)

raw_p, norm_p

(WindowsPath('C:/Users/quantbase/Desktop/fineas/data/raw/yfinance/ohlcv/interval=1d/symbol=AAPL/year=2022/month=01/part-2022-01.parquet'),
 WindowsPath('C:/Users/quantbase/Desktop/fineas/data/norm/ohlcv/interval=1d/symbol=AAPL/year=2022/month=01/part-2022-01.parquet'))

In [3]:
expected_raw = ROOT / "data" / "raw" / "yfinance" / "ohlcv" / f"interval={interval}" / f"symbol={symbol}" / f"year={year:04d}" / f"month={month:02d}" / f"part-{year:04d}-{month:02d}.parquet"
expected_norm = ROOT / "data" / "norm" / "ohlcv" / f"interval={interval}" / f"symbol={symbol}" / f"year={year:04d}" / f"month={month:02d}" / f"part-{year:04d}-{month:02d}.parquet"

assert raw_p == expected_raw, (raw_p, expected_raw)
assert norm_p == expected_norm, (norm_p, expected_norm)

print("OK: raw + norm path contracts match.")

OK: raw + norm path contracts match.


In [4]:
nr = norm_ohlcv_root(interval, repo_root=ROOT)
nd = norm_ohlcv_part_dir(symbol, interval, year, month, repo_root=ROOT)

nr, nd

(WindowsPath('C:/Users/quantbase/Desktop/fineas/data/norm/ohlcv/interval=1d'),
 WindowsPath('C:/Users/quantbase/Desktop/fineas/data/norm/ohlcv/interval=1d/symbol=AAPL/year=2022/month=01'))

In [ ]:
# Phase 2B -  Norm schema contract (frozen) + QC

In [ ]:
from fineas.io.paths import raw_yf_ohlcv_part_path
from fineas.io.qc import qc_norm_ohlcv_partition

In [10]:
raw_p = raw_yf_ohlcv_part_path("AAPL", "1d", 2022, 1, repo_root=ROOT)
raw_df = pd.read_parquet(raw_p)

qc_raw_as_norm = qc_norm_ohlcv_partition(raw_df, name="raw_as_norm_should_fail")
qc_raw_as_norm.to_dict()

{'ok': False,
 'hard_fails': ["raw_as_norm_should_fail: missing required columns: ['returns', 'movement']"],
 'warnings': [],
 'stats': {'name': 'raw_as_norm_should_fail',
  'rows': 20,
  'missing_cols': ['returns', 'movement']}}

In [11]:
d = raw_df.copy().sort_values(["symbol", "ts"]).reset_index(drop=True)

d["returns"] = d.groupby("symbol")["adj_close"].pct_change(fill_method=None) * 100.0

def classify(r):
    if pd.isna(r):
        return np.nan
    if r > 0.55:
        return "rise"
    if r < -0.5:
        return "fall"
    return "freeze"

d["movement"] = d["returns"].apply(classify)

qc_norm = qc_norm_ohlcv_partition(d, name="norm_like_should_pass")
qc_norm.to_dict()

{'ok': True,
 'hard_fails': [],
 'warnings': [],
 'stats': {'name': 'norm_like_should_pass',
  'rows': 20,
  'key_name': 'norm_like_should_pass/key',
  'key_rows': 20,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'norm_like_should_pass/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'ohlcv_name': 'norm_like_should_pass/ohlcv',
  'ohlcv_rows': 20,
  'ohlcv_ohlc_violation_rows': 0,
  'ohlcv_volume_negative_rows': 0,
  'ohlcv_nan_counts': {'open': 0,
   'high': 0,
   'low': 0,
   'close': 0,
   'volume': 0},
  'returns_mismatch_n': 0,
  'returns_missing_expected_n': 0,
  'movement_mismatch_n': 0,
  'cross_part_unverifiable_returns_n': 0}}

In [ ]:
# Phase 2C - Normalization module (raw → norm + labels)

In [13]:
from fineas.norm.ohlcv import build_norm_partition
from fineas.io.paths import raw_yf_ohlcv_part_path, norm_ohlcv_part_path
from fineas.io.qc import qc_norm_ohlcv_partition

In [14]:
# build one partition

res = build_norm_partition(symbol="AAPL", interval="1d", year=2022, month=1, repo_root=ROOT)
res

NormBuildResult(ok=True, status='ok', symbol='AAPL', interval='1d', year=2022, month=1, rows=20, out_path='C:\\Users\\quantbase\\Desktop\\fineas\\data\\norm\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=01\\part-2022-01.parquet', qc={'ok': True, 'hard_fails': [], 'warnings': [], 'stats': {'name': 'norm_ohlcv:AAPL/1d/2022-01', 'rows': 20, 'key_name': 'norm_ohlcv:AAPL/1d/2022-01/key', 'key_rows': 20, 'key_key_cols': ['symbol', 'ts'], 'key_dup_keys_n': 0, 'time_name': 'norm_ohlcv:AAPL/1d/2022-01/time', 'time_monotonic_bad_symbols': [], 'time_max_gap_days': 4.0, 'time_gap_warn_symbols': [], 'ohlcv_name': 'norm_ohlcv:AAPL/1d/2022-01/ohlcv', 'ohlcv_rows': 20, 'ohlcv_ohlc_violation_rows': 0, 'ohlcv_volume_negative_rows': 0, 'ohlcv_nan_counts': {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0}, 'returns_mismatch_n': 0, 'returns_missing_expected_n': 0, 'movement_mismatch_n': 0, 'cross_part_unverifiable_returns_n': 0}}, details={'raw_path': 'C:\\Users\\quantbase\\Desktop\\fineas\\da

In [15]:
# read norm and QC

norm_p = norm_ohlcv_part_path("AAPL", "1d", 2022, 1, repo_root=ROOT)
df = pd.read_parquet(norm_p)
df.head(), df.dtypes

(                         ts symbol        open        high         low  \
 0 2022-01-03 00:00:00+00:00   AAPL  177.830002  182.880005  177.710007   
 1 2022-01-04 00:00:00+00:00   AAPL  182.630005  182.940002  179.119995   
 2 2022-01-05 00:00:00+00:00   AAPL  179.610001  180.169998  174.639999   
 3 2022-01-06 00:00:00+00:00   AAPL  172.699997  175.300003  171.639999   
 4 2022-01-07 00:00:00+00:00   AAPL  172.889999  174.139999  171.029999   
 
         close   adj_close     volume   returns movement  
 0  182.009995  178.103653  104487900       NaN      NaN  
 1  179.699997  175.843246   99310400 -1.269152     fall  
 2  174.919998  171.165817   94537600 -2.659999     fall  
 3  172.000000  168.308533   96904000 -1.669308     fall  
 4  172.169998  168.474854   86709100  0.098819   freeze  ,
 ts           datetime64[ms, UTC]
 symbol                       str
 open                     float64
 high                     float64
 low                      float64
 close                 

In [16]:
# returns + movement should be there

df[["ts", "adj_close", "returns", "movement"]].head(10)

,ts,adj_close,returns,movement
0,2022-01-03 00:00:00+00:00,178.103653,NaN,NaN
1,2022-01-04 00:00:00+00:00,175.843246,-1.269152,fall
2,2022-01-05 00:00:00+00:00,171.165817,-2.659999,fall
3,2022-01-06 00:00:00+00:00,168.308533,-1.669308,fall
4,2022-01-07 00:00:00+00:00,168.474854,0.098819,freeze
5,2022-01-10 00:00:00+00:00,168.494431,0.011620,freeze
6,2022-01-11 00:00:00+00:00,171.322418,1.678386,rise
7,2022-01-12 00:00:00+00:00,171.762756,0.257023,freeze
8,2022-01-13 00:00:00+00:00,168.494431,-1.902814,fall
9,2022-01-14 00:00:00+00:00,169.355560,0.511073,freeze


In [17]:
# Phase 2D -  run Normalization across (symbols × months) implied by run_spec.start/end

In [18]:
import subprocess

cmd = [sys.executable, str(ROOT / "scripts" / "02_build_norm.py")]
r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)

print("returncode:", r.returncode)
print("stdout:\n", r.stdout[:2000])
print("stderr:\n", r.stderr[:2000])

returncode: 0
stdout:
 [fineas] wrote: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\norm_report.json
[fineas] summary: ok_parts=96/96, qc_failed=0, error=0

stderr:
 


In [19]:
# Load norm_report.json and inspect summary


import json

runs_dir = ROOT / "data" / "meta" / "runs"
latest = sorted([p for p in runs_dir.glob("run=*") if p.is_dir()])[-1]
report_path = latest / "norm_report.json"
print("report:", report_path)

payload = json.loads(report_path.read_text(encoding="utf-8"))
payload["summary"]

report: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\norm_report.json


{'run_id': 'run=20260304T102000Z',
 'created_utc': '2026-03-04T11:19:10.595597+00:00',
 'interval': '1d',
 'start': '2022-01-01',
 'end': '2024-01-01',
 'symbols': ['AAPL', 'MSFT', 'NVDA', 'NEWGEN.NS'],
 'months': [[2022, 1],
  [2022, 2],
  [2022, 3],
  [2022, 4],
  [2022, 5],
  [2022, 6],
  [2022, 7],
  [2022, 8],
  [2022, 9],
  [2022, 10],
  [2022, 11],
  [2022, 12],
  [2023, 1],
  [2023, 2],
  [2023, 3],
  [2023, 4],
  [2023, 5],
  [2023, 6],
  [2023, 7],
  [2023, 8],
  [2023, 9],
  [2023, 10],
  [2023, 11],
  [2023, 12]],
 'expected_parts': 96,
 'ok_parts': 96,
 'missing_raw_parts': 0,
 'empty_raw_parts': 0,
 'missing_cols_raw_parts': 0,
 'qc_failed_parts': 0,
 'error_parts': 0,
 'movement_counts_by_symbol': {'AAPL': {'rise': 181,
   'fall': 179,
   'freeze': 140,
   'nan': 2},
  'MSFT': {'rise': 188, 'fall': 183, 'freeze': 129, 'nan': 2},
  'NVDA': {'rise': 232, 'fall': 211, 'freeze': 57, 'nan': 2},
  'NEWGEN.NS': {'rise': 185, 'fall': 195, 'freeze': 112, 'nan': 2}}}

In [21]:
# Sanity check - ensure at least one norm parquet exists + sample read

ok_items = [x for x in payload["results"] if x.get("status") == "ok"]
len(ok_items), ok_items[0]

(96,
 {'symbol': 'AAPL',
  'year': 2022,
  'month': 1,
  'status': 'ok',
  'ok': True,
  'rows': 20,
  'out_path': 'C:\\Users\\quantbase\\Desktop\\fineas\\data\\norm\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=01\\part-2022-01.parquet',
  'details': {'raw_path': 'C:\\Users\\quantbase\\Desktop\\fineas\\data\\raw\\yfinance\\ohlcv\\interval=1d\\symbol=AAPL\\year=2022\\month=01\\part-2022-01.parquet',
   'prev_adj_close_used': False},
  'qc': {'ok': True,
   'hard_fails': [],
   'warnings': [],
   'stats': {'name': 'norm_ohlcv:AAPL/1d/2022-01',
    'rows': 20,
    'key_name': 'norm_ohlcv:AAPL/1d/2022-01/key',
    'key_rows': 20,
    'key_key_cols': ['symbol', 'ts'],
    'key_dup_keys_n': 0,
    'time_name': 'norm_ohlcv:AAPL/1d/2022-01/time',
    'time_monotonic_bad_symbols': [],
    'time_max_gap_days': 4.0,
    'time_gap_warn_symbols': [],
    'ohlcv_name': 'norm_ohlcv:AAPL/1d/2022-01/ohlcv',
    'ohlcv_rows': 20,
    'ohlcv_ohlc_violation_rows': 0,
    'ohlcv_volume_negative_rows'

In [22]:
p = Path(ok_items[0]["out_path"])
df = pd.read_parquet(p)
df.head(), df.dtypes

(                         ts symbol        open        high         low  \
 0 2022-01-03 00:00:00+00:00   AAPL  177.830002  182.880005  177.710007   
 1 2022-01-04 00:00:00+00:00   AAPL  182.630005  182.940002  179.119995   
 2 2022-01-05 00:00:00+00:00   AAPL  179.610001  180.169998  174.639999   
 3 2022-01-06 00:00:00+00:00   AAPL  172.699997  175.300003  171.639999   
 4 2022-01-07 00:00:00+00:00   AAPL  172.889999  174.139999  171.029999   
 
         close   adj_close     volume   returns movement  
 0  182.009995  178.103653  104487900       NaN      NaN  
 1  179.699997  175.843246   99310400 -1.269152     fall  
 2  174.919998  171.165817   94537600 -2.659999     fall  
 3  172.000000  168.308533   96904000 -1.669308     fall  
 4  172.169998  168.474854   86709100  0.098819   freeze  ,
 ts           datetime64[ms, UTC]
 symbol                       str
 open                     float64
 high                     float64
 low                      float64
 close                 

In [23]:
# recheck QC

import sys
from pathlib import Path

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fineas.io.qc import qc_norm_ohlcv_partition

qc = qc_norm_ohlcv_partition(df, name=f"recheck:{p.name}")
qc.to_dict()

{'ok': True,
 'hard_fails': [],
 'warnings': [],
 'stats': {'name': 'recheck:part-2022-01.parquet',
  'rows': 20,
  'key_name': 'recheck:part-2022-01.parquet/key',
  'key_rows': 20,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'recheck:part-2022-01.parquet/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'ohlcv_name': 'recheck:part-2022-01.parquet/ohlcv',
  'ohlcv_rows': 20,
  'ohlcv_ohlc_violation_rows': 0,
  'ohlcv_volume_negative_rows': 0,
  'ohlcv_nan_counts': {'open': 0,
   'high': 0,
   'low': 0,
   'close': 0,
   'volume': 0},
  'returns_mismatch_n': 0,
  'returns_missing_expected_n': 0,
  'movement_mismatch_n': 0,
  'cross_part_unverifiable_returns_n': 0}}